# KaoLRM — Milestone 1 Geometry Bake-Off 비교 (Colab A100)

Pixel3DMM와 **동일한 입력**으로 비교할 2026 후보. FLAME params + mesh + colored Gaussians.

- 계획: `experiments/milestone1_geometry_bakeoff/README.md`
- **License: 소스 Apache 2.0이나 EG3D/FLAME/weights 때문에 effective use는 비상업 연구.**

> ⚠️ Pixel3DMM과 **다른 Colab runtime**에서 실행 (torch 버전 충돌 방지).
> ⚠️ private 입력/출력은 git 금지. Drive `MyDrive/hair_app` 사용. 명령은 2026-06-21 공식 README 기준.

## 0. GPU 확인

In [ ]:
!nvidia-smi

## 1. conda 설치 (condacolab) — 실행 후 커널 재시작

In [ ]:
!pip -q install condacolab
import condacolab
condacolab.install()

## 2. clone + 환경

In [ ]:
import condacolab; condacolab.check()
import os
%cd /content
if not os.path.exists('/content/KaoLRM'):
    !git clone https://github.com/CyberAgentAILab/KaoLRM.git
%cd /content/KaoLRM
!git rev-parse HEAD

In [ ]:
%%bash
set -e
conda create -n kaolrm python=3.10 -y
conda run -n kaolrm pip install torch==2.9.1 torchvision==0.24.1 --index-url https://download.pytorch.org/whl/cu126
cd /content/KaoLRM
conda run -n kaolrm pip install --no-build-isolation -r requirements.txt
conda run -n kaolrm pip install xformers==0.0.33.post2 --index-url https://download.pytorch.org/whl/cu126

## 3. FLAME 다운로드 (https://flame.is.tue.mpg.de 계정/동의 필요)

In [ ]:
%%bash
set -e
cd /content/KaoLRM
conda run -n kaolrm bash fetch_data.sh

## 4. 사전학습 checkpoint 배치

Releases에서 받아 `releases/mono/`(frontal), `releases/multiview/`(profile)에 둔다.
TODO: https://github.com/CyberAgentAILab/KaoLRM/releases 에서 자산 URL 확인 후 채운다.

In [ ]:
%%bash
set -e
cd /content/KaoLRM
mkdir -p releases/mono releases/multiview
# TODO: wget <RELEASE_ASSET_URL> -O releases/mono/<file>
ls -R releases

## 5. private 입력 준비 (배경 제거) — Drive `hair_app`

KaoLRM은 배경 제거 이미지(OpenLRM 규약)를 받는다. Pixel3DMM과 **같은 원본**을 쓴다.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
INPUT_DIR = '/content/drive/MyDrive/hair_app/inputs'   # Pixel3DMM과 동일 원본 (파일명 자유)
os.makedirs(INPUT_DIR, exist_ok=True)
print('exists:', os.path.exists(INPUT_DIR), '| 개수:', len(os.listdir(INPUT_DIR)))

In [ ]:
%%bash -s "$INPUT_DIR"
set -e
conda run -n kaolrm pip install rembg
mkdir -p /content/kao_inputs_nobg
for f in "$1"/*; do
  [ -f "$f" ] || continue
  conda run -n kaolrm rembg i "$f" "/content/kao_inputs_nobg/$(basename "${f%.*}").png" || true
done
ls /content/kao_inputs_nobg

## 6. 추론

frontal = `infer_mono.sh`, profile = `infer_multiview.sh`.
TODO: 스크립트 입력/출력 경로 인자를 `/content/kao_inputs_nobg` 로 맞춘다(스크립트 내부 변수 확인).

In [ ]:
%%bash
set -e
cd /content/KaoLRM
conda run -n kaolrm sh infer_mono.sh
# conda run -n kaolrm sh infer_multiview.sh   # profile views

## 7. ✅ 3D 결과 미리보기 (인터랙티브)

출력 `.ply` mesh를 노트북에서 바로 회전해 본다. Pixel3DMM 결과와 같은 기준으로 비교.

In [ ]:
!pip -q install trimesh plotly
import glob, trimesh
import plotly.graph_objects as go

cands = glob.glob('/content/KaoLRM/dumps/**/*.ply', recursive=True)
print('찾은 mesh 개수:', len(cands))
for p in sorted(cands)[-10:]:
    print(p)
assert cands, 'mesh 없음 → 6번 추론 셀 정상 종료/출력 경로 확인'

m = trimesh.load(sorted(cands)[-1], force='mesh')
v, f = m.vertices, m.faces
fig = go.Figure(data=[go.Mesh3d(
    x=v[:, 0], y=v[:, 1], z=v[:, 2],
    i=f[:, 0], j=f[:, 1], k=f[:, 2],
    color='lightgray', flatshading=True)])
fig.update_layout(scene=dict(aspectmode='data'), margin=dict(l=0, r=0, t=0, b=0))
fig.show()

## 8. 결과/manifest를 Drive(`hair_app`)에 저장

In [ ]:
import json, subprocess, datetime, pathlib, glob, shutil
commit = subprocess.run(['git','-C','/content/KaoLRM','rev-parse','HEAD'],
                        capture_output=True, text=True).stdout.strip()
manifest = {
    'model': 'kaolrm',
    'commit': commit,
    'license': 'Apache-2.0 code; effective non-commercial (EG3D/FLAME/weights)',
    'input_set_id': 'set01',
    'mode': 'mono',   # or multiview
    'gpu': 'A100',
    'created_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'fixes_applied': [],
}
base = pathlib.Path('/content/drive/MyDrive/hair_app')
(base / 'manifests').mkdir(parents=True, exist_ok=True)
(base / 'results').mkdir(parents=True, exist_ok=True)
(base / 'manifests' / 'kaolrm_set01.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
for p in glob.glob('/content/KaoLRM/dumps/**/*.ply', recursive=True):
    shutil.copy(p, base / 'results' / pathlib.Path(p).name)
print('saved to', base)
print(json.dumps(manifest, indent=2, ensure_ascii=False))

## 9. 점수화

`scoring_sheet.csv`에 Pixel3DMM과 같은 기준으로 기록하고, README Gate에 따라 임시 baseline 선택.